# B2-023 — Practice p12

*50 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** B · **Type:** constrained-coding · **Difficulty:** advanced · **Time budget:** 50 minutes  
**Concepts:** stable-diffusion

## Task

Implement classifier-free guidance and a cross-attention conditioning block with an exact Q/K/V shape ledger.

**Contract.**
`cfg_combine(eps_uncond, eps_cond, w)` returns `eps_uncond + w * (eps_cond - eps_uncond)` for two tensors of the same shape and a Python float `w`; it raises `ValueError` if the shapes differ.
`class CrossAttentionConditioner(nn.Module)` with constructor `CrossAttentionConditioner(query_in, dim, num_classes)` creates, in this order, `self.q_proj = nn.Linear(query_in, dim)`, `self.k_proj = nn.Linear(dim, dim, bias=False)`, `self.v_proj = nn.Linear(dim, dim, bias=False)`, `self.class_tokens = nn.Embedding(num_classes, dim)`, and `self.null_token = nn.Parameter(torch.zeros(dim))`, and stores `self.dim = dim`. `forward(query_feats, labels, keep)` takes float `query_feats` `(B,query_in)`, int64 `labels` `(B,)`, and bool `keep` `(B,)`; it builds `tokens = torch.stack([self.class_tokens(labels), self.null_token.expand(B, dim)], dim=1)` (class token first, shape `(B,2,dim)`), `q = self.q_proj(query_feats).unsqueeze(1)` `(B,1,dim)`, `k = self.k_proj(tokens)`, `v = self.v_proj(tokens)`, `scores = q @ k.transpose(1, 2) / math.sqrt(dim)` `(B,1,2)`, sets the class-token score to `float('-inf')` (with `masked_fill`) for every example whose `keep` is `False`, takes `weights = torch.softmax(scores, dim=-1)`, and returns `(weights @ v).squeeze(1)` of shape `(B,dim)`.
`attention_shape_ledger(block, query_feats, labels, keep)` returns the list
`[("query", s_q), ("tokens", s_tok), ("keys", s_k), ("values", s_v), ("scores", s_s), ("weights", s_w), ("output", s_out)]` of shape tuples observed while running the forward computation (`scores` before masking, `weights` after the softmax).
The block also exposes `attention_weights(query_feats, labels, keep)`, returning the `(B,1,2)` weights.

**Banned:** `nn.MultiheadAttention`, `F.scaled_dot_product_attention`, Python loops over the batch, and "removing" a token by setting its score to $0$.

**Probes** (`ATOL = RTOL = 1e-6` unless stated):

1. **Guidance.** With `U` and `C` below: `cfg_combine(U, C, 0.0)` is `torch.equal` to `U`; `cfg_combine(U, C, 1.0)` equals `C`; `cfg_combine(U, C, 1.5)` equals `[[0.65, 0.35], [-0.5, 1.5]]`. A local sign-flipped variant `U - w * (C - U)` agrees at $w=0$ and fails at $w=1$.
2. **Shape ledger.** After `torch.manual_seed(SEED)`, `block = CrossAttentionConditioner(3, 16, 3)`; with `QF5 = torch.randn(5, 3, generator=torch.Generator().manual_seed(SEED))`, `LABELS5`, and `KEEP5` below, the ledger equals `[("query", (5, 1, 16)), ("tokens", (5, 2, 16)), ("keys", (5, 2, 16)), ("values", (5, 2, 16)), ("scores", (5, 1, 2)), ("weights", (5, 1, 2)), ("output", (5, 16))]`.
3. **Literal block.** `lit = CrossAttentionConditioner(3, 2, 2)` with its parameters overwritten by `set_literal_weights(lit)` (supplied) gives, on `QF`, `LABELS`, `KEEP`, the output `[[1.7751175165, -0.1124412194], [0.6295601130, 0.0556598604], [1.0, -0.5]]`; the dropped row is `torch.equal` to `lit.v_proj(lit.null_token)`.
4. **Weights.** On probe 2's inputs every weight row sums to $1$, and each dropped example's weights are exactly `[0.0, 1.0]`.
5. **Label independence.** Changing the labels of the dropped examples in probe 2 leaves their outputs `torch.equal`; changing a kept example's label changes its output.
6. **Gradient routing.** After `block(QF5, LABELS5, KEEP5).sum().backward()` on probe 2's block, row `2` of `block.class_tokens.weight.grad` is exactly zero (label `2` appears only in a dropped example), while rows `0` and `1` are not.
7. A local "zero-score" variant (class score set to $0$ instead of $-\infty$ for dropped examples) changes the dropped row of probe 3.

**Required answer-check assertions.** Probes 1–7.

In [ ]:
import math
import torch
from torch import nn

SEED = 20261015
U = torch.tensor([[0.2, -0.4], [1.0, 0.0]])
C = torch.tensor([[0.5, 0.1], [0.0, 1.0]])
LABELS5 = torch.tensor([0, 1, 2, 0, 1])
KEEP5 = torch.tensor([True, True, False, True, False])
QF = torch.tensor([[1.0, 2.0, 0.5], [0.0, -1.0, 0.3], [2.0, 0.0, -1.0]])
LABELS = torch.tensor([0, 1, 1])
KEEP = torch.tensor([True, True, False])


def set_literal_weights(block):
    with torch.no_grad():
        block.q_proj.weight.copy_(torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]]))
        block.q_proj.bias.copy_(torch.tensor([0.0, 0.5]))
        block.k_proj.weight.copy_(torch.eye(2))
        block.v_proj.weight.copy_(torch.tensor([[2.0, 0.0], [0.0, 1.0]]))
        block.class_tokens.weight.copy_(torch.tensor([[1.0, 0.0], [0.0, 1.0]]))
        block.null_token.copy_(torch.tensor([0.5, -0.5]))


ATOL = 1e-6
RTOL = 1e-6

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.